In [1]:
import torch, os
os.environ["CUDA_VISIBLE_DEVICES"] = "4" 

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, f1_score, accuracy_score
from tqdm.auto import tqdm

# ==========================================
# 🛑 USER CONFIGURATION
# ==========================================
CONFIG = {
    # 1. 填入模型名稱 (BioBERT, BioClinicalBERT, PubMedBERT...)
    'model_name': 'dmis-lab/biobert-v1.1', 
    
    # 2. 儲存檔名
    'save_name': 'biobert_10fold', 
    
    # --- 統一參數 ---
    'n_folds': 10,          # 10-Fold
    'seed': 42,             # 固定種子
    'max_length': 512,
    'batch_size': 16,       # 若 OOM 改 8
    'epochs': 4,            # 10-Fold 通常跑 3-4 epochs 就夠，避免跑太久
    'learning_rate': 2e-5,
    'data_path': 'final_mimic_dataset.csv',
    'target_column': 'los', 
    'text_column': 'text',  
    'threshold': 7.0
}

# ==========================================
# 核心函式庫 (CORE FUNCTIONS)
# ==========================================
def set_seed(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)

class ICUDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        encoding = self.tokenizer.encode_plus(
            text, add_special_tokens=True, max_length=self.max_len,
            return_token_type_ids=False, padding='max_length',
            truncation=True, return_attention_mask=True, return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler):
    model = model.train()
    losses, correct_predictions = [], 0
    for d in data_loader:
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        targets = d["labels"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, targets)
        
        preds = torch.argmax(outputs.logits, dim=1)
        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())
        
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

def eval_model(model, data_loader, loss_fn, device):
    model = model.eval()
    losses = []
    correct_predictions = 0
    all_preds = []
    all_probs = []    # 新增: 儲存機率
    all_targets = []  # 新增: 儲存真實標籤
    
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["labels"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, targets)
            losses.append(loss.item())
            
            # 取出 Class 1 的機率 (Positive Probability)
            probs = torch.softmax(outputs.logits, dim=1)[:, 1]
            preds = torch.argmax(outputs.logits, dim=1)
            correct_predictions += torch.sum(preds == targets)
            
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            
    metrics = {
        'accuracy': (correct_predictions.double() / len(data_loader.dataset)).item(),
        'loss': np.mean(losses),
        'auc': roc_auc_score(all_targets, all_probs),
        'f1': f1_score(all_targets, all_preds),
    }
    
    # 回傳 metrics 以及原始數據 (用於畫圖)
    return metrics, np.array(all_targets), np.array(all_probs)

def plot_cross_val_roc(history, model_name="Model"):
    plt.figure(figsize=(10, 8))
    
    # 計算平均 ROC
    mean_tpr = np.mean(history['tprs'], axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = auc(history['mean_fpr'], mean_tpr)
    std_auc = np.std(history['aucs'])
    
    plt.plot(history['mean_fpr'], mean_tpr, color='b',
             label=r'Mean ROC (AUC = %0.3f $\pm$ %0.2f)' % (mean_auc, std_auc),
             lw=2, alpha=.8)

    # 畫出標準差陰影
    std_tpr = np.std(history['tprs'], axis=0)
    tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
    tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
    plt.fill_between(history['mean_fpr'], tprs_lower, tprs_upper, color='grey', alpha=.2,
                     label=r'$\pm$ 1 std. dev.')

    # 對角線
    plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r', label='Chance', alpha=.8)

    plt.xlim([-0.05, 1.05])
    plt.ylim([-0.05, 1.05])
    plt.xlabel('False Positive Rate', fontsize=14)
    plt.ylabel('True Positive Rate', fontsize=14)
    plt.title(f'ROC Curve - {model_name}\n(10-Fold Cross-Validation)', fontsize=16)
    plt.legend(loc="lower right", fontsize=12)
    plt.grid(alpha=0.3)
    
    save_file = f"ROC_Curve_{model_name.split('/')[-1]}.png"
    plt.savefig(save_file, dpi=300, bbox_inches='tight')
    print(f"📊 圖表已儲存為: {save_file}")
    plt.show()
    

In [ ]:
# ==========================================
# 主程式
# ==========================================
history = {
    'tprs': [],
    'aucs': [],
    'mean_fpr': np.linspace(0, 1, 100)
}

if __name__ == "__main__":
    set_seed(CONFIG['seed'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🚀 10-Fold CV + ROC  | Model: {CONFIG['model_name']}")
    
    # 1. 讀取資料
    if not os.path.exists(CONFIG['data_path']):
        raise FileNotFoundError(f"找不到 {CONFIG['data_path']}")
    df = pd.read_csv(CONFIG['data_path'])
    df['label'] = (df[CONFIG['target_column']] > CONFIG['threshold']).astype(int)
    df[CONFIG['text_column']] = df[CONFIG['text_column']].fillna("No Report")
    
    X = df[CONFIG['text_column']].to_numpy()
    y = df['label'].to_numpy()
    
    # 2. 初始化
    skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=CONFIG['seed'])
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
    
    fold_results = []
    
    # 初始化畫圖數據容器
    history = {
        'tprs': [],
        'aucs': [],
        'mean_fpr': np.linspace(0, 1, 100)
    }
    
    # 3. 10-Fold 迴圈
    for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
        print(f"\n{'='*20} Fold {fold+1}/{CONFIG['n_folds']} {'='*20}")
        
        X_train, X_val = X[train_index], X[val_index]
        y_train, y_val = y[train_index], y[val_index]
        
        train_ds = ICUDataset(X_train, y_train, tokenizer, CONFIG['max_length'])
        val_ds = ICUDataset(X_val, y_val, tokenizer, CONFIG['max_length'])
        
        train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'])
        
        model = AutoModelForSequenceClassification.from_pretrained(CONFIG['model_name'], num_labels=2)
        model = model.to(device)
        
        optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'])
        total_steps = len(train_loader) * CONFIG['epochs']
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
        loss_fn = nn.CrossEntropyLoss().to(device)
        
        best_fold_f1 = 0
        best_metrics = {}
        # 暫存該 Fold 最佳的 ROC 數據
        best_fold_roc_data = None 
        
        for epoch in range(CONFIG['epochs']):
            train_acc, train_loss = train_epoch(model, train_loader, loss_fn, optimizer, device, scheduler)
            # 取得驗證指標與原始數據
            val_metrics, val_y_true, val_y_prob = eval_model(model, val_loader, loss_fn, device)
            
            print(f"  Ep {epoch+1}: Loss {train_loss:.3f} | Val F1 {val_metrics['f1']:.3f} | AUC {val_metrics['auc']:.3f}")
            
            if val_metrics['f1'] > best_fold_f1:
                best_fold_f1 = val_metrics['f1']
                best_metrics = val_metrics
                # 暫存最佳 Epoch 的預測結果，用於畫圖
                best_fold_roc_data = (val_y_true, val_y_prob)
        
        print(f"✅ Fold {fold+1} Best F1: {best_metrics['f1']:.4f}")
        fold_results.append(best_metrics)
        
        # --- [關鍵] 收集 ROC 數據 ---
        if best_fold_roc_data is not None:
            y_true, y_prob = best_fold_roc_data
            fpr, tpr, _ = roc_curve(y_true, y_prob)
            # 插值法對齊
            interp_tpr = np.interp(history['mean_fpr'], fpr, tpr)
            interp_tpr[0] = 0.0
            history['tprs'].append(interp_tpr)
            history['aucs'].append(best_metrics['auc'])
        
        del model, optimizer, scheduler
        torch.cuda.empty_cache()

    # 4. 總結與畫圖
    print(f"\n{'#'*30}\n🏆 Final Report\n{'#'*30}")
    metrics_df = pd.DataFrame(fold_results)
    print(metrics_df.describe().loc[['mean', 'std']])
    
    # 存檔
    metrics_df.to_csv(f"{CONFIG['save_name']}_metrics.csv", index=False)
    
    # 畫圖
    plot_cross_val_roc(history, model_name=CONFIG['model_name'])